In [5]:
import pandas as pd
import numpy as np
import networkx as nx

# Define the raw, unstructured dataset
data = {
    'user_id': [1, 2, 3, 4],
    'name': ['Rahul', 'Priya', 'Amit', 'Neha'],
    'location': [' New Delhi ', 'Gurgaon', 'delhi', 'Noida'],
    'tech_stack': ['Python, Django', 'React, Node', np.nan, 'Python, Pandas'],
    'friends_list': ['2, 3', '1, 4', '1', '2']
}

df = pd.DataFrame(data)
print("--- Raw Data ---")
display(df.head())

# --- Cleaning & Structuring ---
# Strip whitespace and standardize location names
df["location"] = df["location"].str.strip().str.title()

# Fill missing tech_stack values
df["tech_stack"].fillna("Unknown", inplace=True)

# Convert friends_list into Python lists of integers
df["friends_list"] = df["friends_list"].apply(lambda x: [int(i.strip()) for i in x.split(",")])

print("\n--- Cleaned Data ---")
display(df.head())

# --- Build Graph with NetworkX ---
G = nx.Graph()

# Add nodes (users)
for _, row in df.iterrows():
    G.add_node(row["user_id"], name=row["name"], location=row["location"], tech_stack=row["tech_stack"])

# Add edges (friendships)
for _, row in df.iterrows():
    for friend in row["friends_list"]:
        G.add_edge(row["user_id"], friend)

print("\nGraph Info:")
print(nx.info(G))

# Example: Jaccard similarity for "People You May Know"
print("\nFriend Recommendations (Jaccard Coefficient):")
for u, v, p in nx.jaccard_coefficient(G):
    print(f"{u} - {v}: {p:.2f}")


ModuleNotFoundError: No module named 'networkx'

In [ ]:
# 1. Standardize text and group NCR locations
df['location'] = df['location'].str.strip().str.lower()
df['location'] = df['location'].replace({'new delhi': 'delhi', 'gurgaon': 'ncr', 'noida': 'ncr'})

# 2. Handle missing tech stacks
df['tech_stack'] = df['tech_stack'].fillna('not specified')

# 3. Parse strings into lists for friends and skills
# Adding a check to ensure we only parse valid digits for friends
df['friends_list'] = df['friends_list'].apply(
    lambda x: [int(i.strip()) for i in str(x).split(',') if i.strip().isdigit()] if pd.notna(x) else []
)
df['tech_stack'] = df['tech_stack'].apply(
    lambda x: [skill.strip().lower() for skill in str(x).split(',')]
)

print("\n--- Cleaned and Structured Data ---")
display(df.head())

In [ ]:
# Initialize the network graph
G = nx.Graph()

# Populate nodes and edges based on friends lists
for index, row in df.iterrows():
    user = row['user_id']
    for friend in row['friends_list']:
        G.add_edge(user, friend)

def recommend_friends(graph, target_user_id):
    """Recommends friends based on the volume of mutual connections."""
    recommendations = {}
    
    # Check if the user is in the graph
    if target_user_id not in graph:
        return []
        
    target_friends = set(graph.neighbors(target_user_id))
    
    for node in graph.nodes():
        if node != target_user_id and node not in target_friends:
            node_friends = set(graph.neighbors(node))
            mutual_friends = target_friends.intersection(node_friends)
            
            if mutual_friends:
                recommendations[node] = len(mutual_friends)
                
    # Return a list of tuples sorted by highest mutual friends
    return sorted(recommendations.items(), key=lambda item: item[1], reverse=True)

print("\n--- Friend Recommendations ---")
# Let's test it for User 1
recs_user_1 = recommend_friends(G, target_user_id=1)
print(f"User 1 (Rahul) should connect with User IDs: {[user for user, score in recs_user_1]}")

In [ ]:
# Define the pages and their associated tags
pages = {
    'P1 (Python Delhi)': ['python', 'django', 'data science'],
    'P2 (Frontend NCR)': ['react', 'javascript', 'css'],
    'P3 (Data Analytics Hub)': ['python', 'pandas', 'sql']
}

def recommend_pages(user_skills, pages_dict):
    """Recommends pages using Jaccard Similarity between user skills and page tags."""
    page_scores = {}
    user_set = set(user_skills)
    
    for page_name, tags in pages_dict.items():
        page_set = set(tags)
        
        intersection = len(user_set.intersection(page_set))
        union = len(user_set.union(page_set))
        
        if union > 0:
            similarity = round(intersection / union, 2)
            if similarity > 0: 
                page_scores[page_name] = similarity
                
    return sorted(page_scores.items(), key=lambda item: item[1], reverse=True)

print("\n--- Page Recommendations ---")
# Let's test it for User 4 (Neha), extracting her skills from the cleaned dataframe
user_4_skills = df.loc[df['user_id'] == 4, 'tech_stack'].iloc[0]
page_recs = recommend_pages(user_4_skills, pages)

print(f"User 4 (Neha) Skills: {user_4_skills}")
print("Recommended Pages (with similarity scores):")
for page, score in page_recs:
    print(f"- {page}: {score}")